In [1]:
from datasets import load_dataset
from transformers import ViTFeatureExtractor, AutoModel
from modeling_vit import ViTForImageClassification, ViTSelfAttention
from transformers import TrainingArguments, Trainer
from open_clip_vit import VisionTransformer, MultiheadAttention, to_dist
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from avalanche.benchmarks.classic import SplitCIFAR100, SplitTinyImageNet, SplitCUB200, SplitImageNet
from custom_datasets import SplitImageNetR
from vil_datasets import build_continual_dataloader

from PIL import Image
from progbar import Progbar
from copy import deepcopy
import numpy as np
import open_clip
import argparse
import os
from localdatasets import make_VLCS, make_TI
from margin_loss import LargeMarginLoss
import cv2
import numpy as np
from torchvision.transforms import *


args = argparse.Namespace()
args.dataset = 'DomainNet-dil'
args.adapters_per_domain = 3
args.batch_size = 128
args.num_classes = 345
args.base_model = 'laion'
args.test_domain = 0
args.epochs = 1
args.lr = 1e-3
args.separation_function = 'softmax'
args.include_seen_domain = False

if args.separation_function == 'affine' or args.separation_function is None:
    args.separation_function = to_dist
elif args.separation_function == 'softmax':
    args.separation_function = F.softmax
elif args.separation_function == 'relu':
    args.separation_function = lambda x, dim: F.relu(x)
elif args.separation_function == 'sigmoid':
    args.separation_function = lambda x, dim: torch.sigmoid(x)
elif args.separation_function == 'tanh':
    args.separation_function = lambda x, dim: torch.tanh(x)

dataset_name = args.dataset
is_hf_dataset = True
# dataset = load_dataset("wltjr1007/DomainNet")
# train_domains = [0, 1, 2, 3, 4, 5]
# if not args.include_seen_domain:
#     train_domains.remove(int(args.test_domain))
# test_domain = int(args.test_domain)
# args.num_tasks = 6
# adapter_list = torch.load('/data/ai22mtech12002/projects/WeightDG/weights/train_domain_adapters_list_laion_DomainNet.pt')
# dataset = load_dataset("flwrlabs/pacs")
# train_domains = ['art_painting', 'cartoon', 'photo', 'sketch']
# # train_domains.remove(args.test_domain)
# test_domain = args.test_domain
# adapter_list = torch.load('/data/ai22mtech12002/projects/WeightDG/weights/train_domain_adapters_list_laion_PACS.pt')


# laion, preprocess_train, preprocess_val = open_clip.create_model_and_transforms('hf-hub:laion/CLIP-ViT-B-16-laion2B-s34B-b88K')
# vit = laion.visual.cuda()

@torch.no_grad()
def get_store_dict(model):
    module_weight_dict = {}
    for i in range(12):
        module = model.transformer.resblocks[i]
        lorank_values = []
        for n, p in module.named_parameters():
            if "lora" in n:
                lorank_values.append(p.reshape(-1).detach())
        lorank_values = torch.cat(lorank_values, dim=0)
        module_weight_dict[i] = lorank_values
    return module_weight_dict

@torch.no_grad()
def set_store_dict(model, weight_dict):
    weight_dict = deepcopy(weight_dict)
    for i in range(12):
        module = model.transformer.resblocks[i]
        for n, p in module.named_parameters():
            if "lora" in n:
                p.data = weight_dict[i][:p.numel()].reshape(p.shape)
                weight_dict[i] = weight_dict[i][p.numel():]

def load_laion_weights(vit, laion_vit):
    vit_state_dict = vit.state_dict()
    laion_vit_state_dict = laion_vit.state_dict()
    for n, p in vit_state_dict.items():
        if n in laion_vit_state_dict:
            vit_state_dict[n] = laion_vit_state_dict[n]

    vit.load_state_dict(vit_state_dict)


/data/ai22mtech12002/anaconda3/envs/memcl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@torch.no_grad()
def mixup(image_batch, mixup_times=1):
    alpha = 0.4
    for i in range(mixup_times):
        lam = np.random.beta(alpha, alpha)
        rand_perm = torch.randperm(image_batch.size(0))
        image_batch = lam * image_batch + (1 - lam) * image_batch[rand_perm]
    return image_batch

In [3]:

class DomainDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, preprocess, returns_domain=True, label_offset = None):
        self.dataset = dataset
        self.preprocess = preprocess
        self.label_offset = label_offset
        self.returns_domain = returns_domain

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        # print(list(item['image'].keys()))
        # item['image'].verify()
        if is_hf_dataset:
            if args.base_model == 'vit-in21k':
                item['image'] = self.vit_preprocess(item['image'])
            else:
                item['image'] = self.preprocess(item['image'])
            return {'image': item['image'], 'label': item['label']}
        else:
            if self.returns_domain:
                image, label, _ = item
            else:
                image, label = item
            item = {}
            if args.base_model == 'vit-in21k':
                item['image'] = self.vit_preprocess(image)
            else:
                item['image'] = self.preprocess(image)
            item['label'] = (label - self.label_offset) if self.label_offset is not None else label
            return item


In [4]:
args.num_tasks = 6
args.data_path = '/data/ai22mtech12002/projects/WeightDG/data/DomainNet-dil'
args.task_type = 'dil'
args.shuffle = True
args.versatile_inc = False
args.num_workers = 8
args.pin_mem = True
args.batch_size = 128
preprocess_train = Compose([
        RandomResizedCrop(size=(224, 224), scale=(0.9, 1.0), ratio=(0.75, 1.3333), interpolation=InterpolationMode.BICUBIC, antialias=True),
        ToTensor(),
        Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711]),
    ])
preprocess_val = Compose([
        Resize(size=(256, 256), interpolation=InterpolationMode.BICUBIC, antialias=True),
        CenterCrop((224, 224)),
        ToTensor(),
        Normalize(mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711]),
    ])
dataloaders, _, _ = build_continual_dataloader(args=args)
train_domains = list(range(args.num_tasks))
is_hf_dataset = False
# adapter_list = torch.load('/data/ai22mtech12002/projects/WeightDG/weights/train_domain_adapters_list_laion_DomainNet-dil.pt')
# adapter_list = torch.load('/data/ai22mtech12002/projects/WeightDG/weights/train_domain_adapters_list_laion_DomainNet-dil_stage1_saksham.pt', weights_only=False)


############## dil


In [5]:
test_domain_loaders = []
for domain_idx, domain in enumerate(train_domains):
    if args.dataset in ['iDigits-dil', 'CORe50-dil', 'DomainNet-dil']:
        dl = dataloaders[domain]
        train_dataset = dl['train']
        train_dataset = DomainDataset(train_dataset, preprocess_train, returns_domain=False)
        train_loader = DataLoader(train_dataset, batch_size=args.batch_size, shuffle=True, num_workers=8, pin_memory=True)
        test_dataset = dl['test']
        test_dataset = DomainDataset(test_dataset, preprocess_val, returns_domain=False)
        test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=8, pin_memory=True)
        test_domain_loaders.append(test_loader)
        print(f"Training samples for domain {domain}: {len(train_dataset)}")
        print(f"Testing samples for domain {domain}: {len(test_dataset)}")

Training samples for domain 0: 34019
Testing samples for domain 0: 14814
Training samples for domain 1: 37087
Testing samples for domain 1: 16114
Training samples for domain 2: 52867
Testing samples for domain 2: 22892
Training samples for domain 3: 120750
Testing samples for domain 3: 51750
Training samples for domain 4: 122563
Testing samples for domain 4: 52764
Training samples for domain 5: 49115
Testing samples for domain 5: 21271


In [ ]:
import pickle as pkl
parent_dir = f'data/{dataset_name}'
args.name_tag = ''
model = pkl.load(open(f'{parent_dir}/{args.base_model}_{args.dataset}_{args.name_tag}.pkl', 'rb'))

def get_model_keys(model):
    model_keys = []
    for name, module in model.named_modules():
        if isinstance(module, MultiheadAttention):
            if module.use_hopfield:
                module_keys = module.hopfield_keys
                model_keys.append(module_keys)
                print('Number of keys in module:', name, module.hopfield_keys.shape)
    return model_keys


model_keys = get_model_keys(model)

for name, module in model.named_modules():
    if isinstance(module, MultiheadAttention):
        module.store_hopfield_similarities = True
        print(module.hopfield_keys.shape)

sim_sep_dict = {'sim': {}, 'sep': {}}
@torch.no_grad()
def eval():
    model.eval()
    domain_accs = {}
    for domain_idx, test_loader in enumerate(test_domain_loaders):
        sim_sep_dict['sim'][domain_idx] = {}
        sim_sep_dict['sep'][domain_idx] = {}
        pbar = Progbar(len(test_loader))

        for layer in range(12):
            sim_sep_dict['sim'][domain_idx][layer] = []
            sim_sep_dict['sep'][domain_idx][layer] = []

        for step, batch in enumerate(test_loader):
            if step == 5:
                print('--------------')
                break
            pixel_values = batch['image'].cuda()
            labels = batch['label'].cuda()
            outputs = model(pixel_values)

            pbar.update(step + 1)
            layer = 0
            # for _, (name, module) in enumerate(model.named_modules()):
            #     if isinstance(module, MultiheadAttention):
            #         if layer == 0:
            #             print(module.sim_scores)
            #         sim_sep_dict['sim'][domain_idx][layer].append(module.sim_scores.cpu().numpy())    
            #         sim_sep_dict['sep'][domain_idx][layer].append(module.sep_scores.cpu().numpy())
            #         layer += 1

eval()
# eval_accs = eval()
# print("Final eval accs: ", eval_accs)

torch.Size([768, 3])
torch.Size([768, 3])
torch.Size([768, 3])
torch.Size([768, 3])
torch.Size([768, 3])
torch.Size([768, 3])
torch.Size([768, 3])
torch.Size([768, 3])
torch.Size([768, 3])
torch.Size([768, 3])
torch.Size([768, 3])
torch.Size([768, 3])

tensor([[0.0551, 0.0083, 0.0478]], device='cuda:0')


tensor([[ 0.0197,  0.0136, -0.0176]], device='cuda:0')


tensor([[ 0.0123, -0.0025, -0.0154]], device='cuda:0')


tensor([[-0.0189,  0.0655,  0.0062]], device='cuda:0')


tensor([[ 0.0119, -0.0030, -0.0046]], device='cuda:0')


tensor([[ 0.0206, -0.0070, -0.0004]], device='cuda:0')


tensor([[0.0107, 0.0790, 0.0519]], device='cuda:0')


tensor([[0.0083, 0.0214, 0.0230]], device='cuda:0')


tensor([[-0.0204, -0.0239,  0.0572]], device='cuda:0')


tensor([[-0.0260,  0.0216,  0.0431]], device='cuda:0')


tensor([[ 0.0499, -0.0067, -0.0123]], device='cuda:0')


tensor([[ 0.0502, -0.0182,  0.0642]], device='cuda:0')

    1/14814 [..............................] - ETA: 2:04:16
tensor([[0.05

In [7]:
# Visualize sim and sep scores as a heatmap
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import matplotlib.cm as cm
import matplotlib.colors as mcolors


def plot_heatmap(data, title, xlabel, ylabel):
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(data, annot=True, fmt=".2f", cmap="YlGnBu", cbar=True, ax=ax)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    plt.close()

In [8]:
sim_sep_dict['sep'][0][0]

[]

In [9]:
from matplotlib import pyplot as plt
for domain_idx, domain_dict in sim_sep_dict['sim'].items():
    layer = 0
    sims = np.concatenate(domain_dict[layer], axis=0)
    values = np.abs(sims.mean(axis=0))
    print(values)
    # values = torch.softmax(torch.tensor(values), dim=-1).numpy()

    # plot magnitudes
    plt.figure(figsize=(10, 6))
    plt.bar(range(len(values)), values)
    # plt.title(f"Mean Sim Scores for {k}")
    plt.xlabel("Key")
    plt.ylabel("Mean Sim Score")
    plt.show()

ValueError: need at least one array to concatenate